# Rung 1 — predicting a cell line, or a drug, the model has never seen

**Task** `rung1-held-out-prediction` · **Spec** [design.md](design.md) ·
**Decisions** [decisions.md](decisions.md) · **Verification** [verification.md](verification.md) ·
**Claim-by-claim recomputation** [verify.ipynb](verify.ipynb)

Run this notebook top to bottom. Every number below is read from a table the run wrote, so what
you see is what the artifacts say — not what anyone typed. Point it at a run by setting
`RUNG1_TASK_DIR` to the folder holding its tables and `RUNG1_CACHE` to the scratch cache holding
its per-round and per-block files.

---

## The question

Tahoe-100M measured how hundreds of drugs change gene expression in 50 cancer cell lines. Rung 1
asks whether a model can predict that change for a line, or a drug, it has never seen.

## The two hypotheses, as the design states them

- **H1.** Stack's summary of a cell line — its *embedding*, computed from untreated cells — carries
  what that line in particular does in response to a drug. It should beat **(a)** the drug's
  average effect in the other lines, which knows nothing about this line, and **(b)** the same
  model given the line's ordinary gene expression instead of the embedding.
- **H2.** A Stack fine-tuned on drug experiments (sci-Plex) should predict better than the
  released checkpoint, which was tuned on immune-signalling experiments (cytokines).

We report estimates, 95% confidence intervals, p-values, and the smallest effect each comparison
could reliably detect (its *minimum detectable effect*, MDE). **Nothing passes or fails**
(design section 1): a comparison whose interval crosses zero is reported as such, beside the
effect it could have detected.


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    return start


REPO = find_repo(Path.cwd().resolve())
TASK = "rung1-held-out-prediction"
TASK_DIR = Path(os.environ.get("RUNG1_TASK_DIR") or (REPO / "docs" / "tasks" / TASK))
CACHE_DIR = Path(os.environ["RUNG1_CACHE"]) if os.environ.get("RUNG1_CACHE") else None
FIG = TASK_DIR / "figures"

SCHEME_NAMES = {"lolo": "a line hidden", "lodo": "a drug hidden"}


def table(name, directory=None):
    """A table the run wrote, or a plain note that the run has not happened yet.

    keep_default_na=False is not a detail: one of the fifty cell lines has no DepMap identifier
    upstream and appears as the literal string "NA", which pandas would otherwise read as a
    missing value and drop from every count below. float_precision="round_trip" keeps the
    reader from moving the last bit of a double.
    """
    path = (directory or TASK_DIR) / name
    if not path.exists():
        print(f"[not present yet] {path}")
        return None
    return pd.read_csv(path, keep_default_na=False, na_values=[""], float_precision="round_trip")


def record(name, directory=None):
    path = (directory or TASK_DIR) / name
    if not path.exists():
        print(f"[not present yet] {path}")
        return None
    return json.loads(path.read_text())


def show(name, caption=""):
    path = FIG / name
    if not path.exists():
        print(f"[figure not present yet] {path}")
        return
    if caption:
        print(caption)
    display(Image(filename=str(path)))


def find_grid():
    """The grid record, from the task folder or the run's cache -- both are legitimate."""
    for directory in (TASK_DIR, CACHE_DIR):
        if directory is None:
            continue
        path = directory / "rung1_grid.json"
        if path.exists():
            return json.loads(path.read_text())
    print(f"[not present yet] {TASK_DIR / 'rung1_grid.json'}")
    return None


grid = find_grid()
ceiling = table("rung1_ceiling.csv")
model_summary = table("rung1_model_summary.csv")
comparisons = table("rung1_comparisons.csv")
settings = table("rung1_settings.csv")
pair_scores = table("rung1_pair_scores.csv.gz")
params = record("rung1_run.params.json")

print("run present:", model_summary is not None)
if params:
    print("\n### Provenance of what you are reading")
    print(f"  {'commit':16s} {params['git_sha']}")
    print(f"  {'slurm job':16s} {params['slurm_job_id']}")
    print(f"  {'grid':16s} {params['n_lines']} lines x {params['n_drugs']} drugs")
    print(
        f"  {'redraws':16s} {params['n_draws']} in {params['n_blocks']} blocks, "
        f"base seed {params['seeds']['redraw_base_seed']}"
    )
    print(f"  {'ceiling (√SB)':16s} {params['ceiling']}")
    print(f"  {'tranche':16s} tahoe100m-pseudobulk-de.v1, the same table rung 0 measured")

## What stays fixed

One dose, one complete rectangle of lines and drugs, one score, and a ceiling inherited from
rung 0 rather than remeasured.

| Item | Choice | Why |
|---|---|---|
| Dose | **5 micromolar only** | the dose with by far the most repeated measurements |
| Pairs | every drug measured in **every** line, at that dose | no gaps, so nothing about a line can be inferred from which drugs it happens to have |
| Answer | each gene's change in expression (`log2FoldChange`), averaged over the pair's plates | all of the pair's data |
| Score | correlation across genes between predicted and measured change, per pair, averaged over pairs | rung 0's score |
| Genes | **responding** (called changed on at least one of the pair's plates) and **all** | responding genes carry the repeatable signal; rung 0 measured 0.58 against 0.08 |
| Ceiling | rung 0's promoted 5 uM reliability, limited to these pairs, square-rooted | a prediction meets measurement noise once; two measurements meet it twice |


In [ ]:
if grid:
    excluded = [tuple(pair) for pair in grid["excluded_pairs"]]
    print(
        f"lines: {len(grid['lines'])}   drugs: {len(grid['drugs'])}   "
        f"pairs: {len(grid['lines']) * len(grid['drugs']):,}"
    )
    print(f"dose: {grid['dose']} uM")
    print(f"pairs removed because a model had already seen them: {len(excluded)} {excluded}")

if ceiling is not None:
    display(ceiling.set_index("gene_set"))
    print("\nThe ceiling is √SB, not SB. A measurement is the true change plus noise; two")
    print("measurements of it correlate at the reliability R, but the truth correlates with one")
    print("measurement at √R. A prediction is not a second measurement, so only one side is")
    print("noisy -- dividing by R would let a perfect model score over 100%.")

## Step 1 — build: what every model is told about a cell line

Each line is described once, from untreated cells given only the solvent (DMSO) — never from
anything the drugs did. Up to 1,000 cells per line, spread evenly over its plates, picked with a
fixed seed. Six descriptions: the line's average expression; two compressions of it (principal
components, and non-negative factors); and the average Stack embedding of its cells under three
checkpoints — the pretrained one, the released cytokine-tuned one, and our sci-Plex fine-tune.

**The control.** A description built from half a line's cells should identify that line among all
fifty when matched against the other halves, and should not when the cells' line labels are
shuffled first. The table beside the figure is that check, description by description, with the
shuffled null's 99th percentile as the bar to clear. The last row is the weights check that H2
rests on: the drug fine-tune's encoder must actually differ from the base checkpoint, or the two
descriptions would be the same numbers.


In [ ]:
cells = table("rung1_cells.csv")
if cells is not None:
    print(
        f"cells selected per line: median {cells['n_selected'].median():,.0f}, "
        f"total {cells['n_selected'].sum():,}"
    )
    print(f"plates per line: median {cells.groupby('line').size().median():.0f}")

build = table("rung1_control_build.csv")
if build is not None:
    identity = build.loc[build["check"] == "half_vs_half_identity"]
    display(identity[["subject", "identity_share", "null_mean", "null_p99", "above_null_p99"]])
    weights = build.loc[build["check"] == "drug_finetune_weights_differ"]
    if len(weights):
        row = weights.iloc[0]
        print(
            f"\ndrug fine-tune against the base checkpoint: "
            f"{int(row['n_different_tensors'])} of {int(row['n_shared_tensors'])} shared "
            f"tensors differ; encoders identical: {row['encoders_identical']}"
        )
show("01_build.png")

## Step 2 — split: how a line, or a drug, is hidden

**Leave one line out** (the main test, 50 rounds): each round hides one cell line; every model
learns from the other 49 and predicts all 107 drugs for the hidden line, seeing only how that
line looks untreated. **Leave one drug out** (the second test, 107 rounds): each round hides one
drug; models learn from the other 106 and predict it in every line, seeing only its chemical
structure. A line and a drug are never hidden together, and every setting is chosen inside the
round from training data alone.

**The control.** A signal planted in one unit's own answers and nowhere else must be recovered by
a deliberately broken split that leaves the hidden unit in training, and must come back at zero
under the split the run actually uses. The figure's first panel is the grid of rounds as it was
scored — the gaps are the pairs no model could be scored on, and the two removed for leakage.


In [ ]:
if pair_scores is not None:
    for scheme, label in SCHEME_NAMES.items():
        rows = pair_scores.loc[pair_scores["scheme"] == scheme]
        if rows.empty:
            continue
        by_gene_set = rows.groupby("gene_set").apply(
            lambda part: pd.Series(
                {
                    "rounds": part["round"].nunique(),
                    "models": part["model"].nunique(),
                    "pairs scored": part.groupby(["line", "drug"]).ngroups,
                    "scores": len(part),
                }
            ),
            include_groups=False,
        )
        print(f"--- {label} ---")
        display(by_gene_set)

split = table("rung1_control_split.csv")
if split is not None:
    columns = [
        "scheme",
        "split",
        "estimate",
        "ci_lo",
        "ci_hi",
        "p",
        "mde",
        "detected",
        "within_mde",
    ]
    display(split[columns])
    print("\nThe broken split finds the planted signature; the shipped split scores it within")
    print("its own minimum detectable effect of zero -- the split does not leak the answer.")
show("02_split.png")

## Step 3 — fit: one estimator, six descriptions, and the settings chosen inside each round

Every description goes through the same model — a ridge regression, a linear fit held back from
chasing noise — so that only the description changes. With a line hidden, it learns how the 49
training lines' departures from the drug average follow their descriptions, predicts the hidden
line's departure, and adds the average back. With a drug hidden, it learns from all training
pairs at once, letting chemically similar drugs share effects, more strongly in lines whose
descriptions are alike. Two references stand in for "knowing nothing about the line": the drug's
average effect in the training lines, and (with a drug hidden) a chemistry-only model. A
**nearest-lines** model averages the k training lines that look most like the hidden one. Each
description also runs against a **random stand-in** of the same width, through the same fit — so
a description that beats its stand-in gained from what it describes, not from the method.

How hard the fit is held back, and how many components a compression keeps, are chosen inside
each round by leaving out one training line (or drug) at a time — never using the hidden unit.

**The control.** On synthetic data of the screen's size and noise, a line-specific response
planted at a known strength must be recovered by the matching description and not by its random
stand-in; with nothing planted, no model may gain beyond its MDE, and the fit must sit at the
most conservative penalty it offers.


In [ ]:
if settings is not None:
    chosen = settings.copy()
    chosen["lambda"] = pd.to_numeric(chosen["lambda"], errors="coerce")
    chosen["k"] = pd.to_numeric(chosen["k"], errors="coerce")
    summary = chosen.groupby(["scheme", "model"]).agg(
        rounds=("round", "nunique"),
        median_penalty=("lambda", "median"),
        median_components=("k", "median"),
        penalty_at_edge=("lambda_at_edge", "mean"),
    )
    display(summary)
    if params:
        print(f"\ncandidate component counts the run tuned over: {params['component_ks']}")

fit = table("rung1_control_fit.csv")
if fit is not None:
    planted = fit.loc[fit["planted"]]
    columns = ["scheme", "contrast", "estimate", "ci_lo", "ci_hi", "p", "mde", "ratio_to_mde"]
    display(planted[columns])
    nothing = fit.loc[~fit["planted"]]
    print("\nwith nothing planted (the negative control):")
    display(nothing[["scheme", "contrast", "estimate", "mde", "lambda_at_top_share"]])
show("03_fit.png")

## Step 4 — score: how well each model did, against the best anything could do

Each model's score is the average, over the pairs every model could be scored on, of the
correlation across genes between what it predicted and what was measured. Beside it, that score
as a fraction of the ceiling √SB — what a perfect predictor of the true change would score
against this noisy measurement.

**The control.** Synthetic answers are built at a known reliability R, and put through the same
scoring code: the true change must score √R and a second noisy measurement must score R. That is
what makes the ceiling a measurement rather than an assertion.


In [ ]:
if model_summary is not None:
    for scheme, label in SCHEME_NAMES.items():
        rows = model_summary.loc[
            (model_summary["scheme"] == scheme) & (model_summary["gene_set"] == "responding")
        ].sort_values("mean_r", ascending=False)
        if rows.empty:
            continue
        print(f"--- {label}, responding genes ---")
        display(
            rows.set_index("model")[
                ["n_pairs", "mean_r", "ci_lo", "ci_hi", "mde", "sqrt_sb", "fraction_of_ceiling"]
            ]
        )
    print("the same models on all genes, for reference:")
    allgenes = model_summary.loc[model_summary["gene_set"] == "all"].sort_values(
        ["scheme", "mean_r"], ascending=[True, False]
    )
    indexed = allgenes.set_index(["scheme", "model"])
    display(indexed[["mean_r", "ci_lo", "ci_hi", "fraction_of_ceiling"]])

score = table("rung1_control_score.csv")
if score is not None:
    columns = ["reliability", "gene_set", "prediction", "planted", "mean_r", "se", "within_3_se"]
    display(score[columns])
show("04_score.png")

## Step 5 — null: what the comparisons could and could not have detected

A comparison is the average, over the pairs both models scored, of one model's score minus the
other's. Its uncertainty comes from redrawing the held-out units with repeats — the 50 lines when
a line is hidden, the 107 drugs when a drug is hidden — 2,000 times and recomputing the
difference each time. The middle 95% of those redraws is the interval; twice the smaller share of
them past zero is the p-value; 2.8 of their standard deviations is the minimum detectable effect,
the smallest true difference a two-sided 5% test would find 80% of the time. Because six
comparisons are read under the main test and five under the second, p-values are adjusted by
Holm's method **within each test**.

**The control.** A comparison shifted by exactly its own MDE must be detected about 80% of the
time, and with nothing shifted, at most 5% — the two rates an MDE is a statement about.


In [ ]:
if comparisons is not None:
    declared = comparisons.loc[comparisons["hypothesis"].astype(str).str.len() > 0]
    for scheme, label in SCHEME_NAMES.items():
        rows = declared.loc[(declared["scheme"] == scheme) & (declared["gene_set"] == "responding")]
        if rows.empty:
            continue
        print(f"--- {label}, responding genes (the design's test) ---")
        display(
            rows.set_index("comparison")[
                ["hypothesis", "n_pairs", "estimate", "ci_lo", "ci_hi", "p", "p_holm", "mde"]
            ]
        )

    stand_ins = comparisons.loc[
        (comparisons["hypothesis"].astype(str).str.len() == 0)
        & (comparisons["gene_set"] == "responding")
    ]
    if len(stand_ins):
        print("each description against its own random stand-in (reported unadjusted):")
        display(
            stand_ins.set_index("comparison")[["scheme", "estimate", "ci_lo", "ci_hi", "p", "mde"]]
        )

    print("\npairs share lines and drugs, so each comparison is also redrawn over lines and")
    print("drugs together; the design effect is how much that inflates its variance, and the")
    print("width ratio how much wider the interval gets:")
    display(
        comparisons.loc[comparisons["gene_set"] == "responding"]
        .set_index("comparison")[["design_effect", "width_ratio"]]
        .describe()
        .loc[["mean", "min", "max"]]
    )

null = table("rung1_control_null.csv")
if null is not None:
    columns = ["check", "target_rate", "repetitions", "detections", "rate", "ci_lo", "ci_hi"]
    display(null[[*columns, "inside_interval"]])
show("05_null.png")

## What the run says about the two hypotheses

Each hypothesis is read as the estimates below, with their intervals and the effect each
comparison could have detected. A difference whose interval crosses zero is a difference this
run could not separate from no difference — which is a result, not a failure.


In [ ]:
def read_hypothesis(label):
    """Every declared comparison carrying `label`, on responding genes, as plain sentences."""
    if comparisons is None:
        return
    rows = comparisons.loc[
        (comparisons["hypothesis"] == label) & (comparisons["gene_set"] == "responding")
    ]
    if rows.empty:
        print(f"no {label} rows in this run")
        return
    for _, row in rows.iterrows():
        interval = f"95% CI {float(row['ci_lo']):+.4f} to {float(row['ci_hi']):+.4f}"
        holm = float(row["p_holm"]) if str(row["p_holm"]).strip() else float("nan")
        excludes_zero = float(row["ci_lo"]) * float(row["ci_hi"]) > 0
        separated = "excludes zero" if excludes_zero else "includes zero"
        print(
            f"{SCHEME_NAMES[str(row['scheme'])]}: {row['model_a']} - {row['model_b']} = "
            f"{float(row['estimate']):+.4f} ({interval}); p {float(row['p']):.4f}, "
            f"Holm {holm:.4f}; MDE {float(row['mde']):.4f}; the interval {separated}."
        )
    print()


print("H1(a) -- the embedding against a model that knows nothing about the line:")
read_hypothesis("H1(a)")
print("H1(b) -- the embedding against the same model given ordinary expression, its compressions,")
print("or the most similar training lines:")
read_hypothesis("H1(b)")
print("H2 -- the drug fine-tune against the released cytokine-tuned checkpoint:")
read_hypothesis("H2")

## Conclusions

Read as estimates with uncertainty. **Nothing here passes or fails** — the design says so, and a
comparison that did not separate two models is reported beside the smallest difference it could
have separated, so an inconclusive result cannot be mistaken for a null one.


In [ ]:
if model_summary is not None and comparisons is not None:
    for scheme, label in SCHEME_NAMES.items():
        rows = model_summary.loc[
            (model_summary["scheme"] == scheme) & (model_summary["gene_set"] == "responding")
        ]
        if rows.empty:
            continue
        best = rows.loc[rows["mean_r"].idxmax()]
        reference = "drug_average" if scheme == "lolo" else "chemistry_only"
        baseline = rows.loc[rows["model"] == reference]
        print(f"--- {label}, responding genes ---")
        print(
            f"  best score: {best['model']} at {float(best['mean_r']):.4f} "
            f"(95% CI {float(best['ci_lo']):.4f} to {float(best['ci_hi']):.4f}), "
            f"{float(best['fraction_of_ceiling']):.1%} of the ceiling √SB "
            f"{float(best['sqrt_sb']):.4f}"
        )
        if len(baseline):
            row = baseline.iloc[0]
            fraction = float(row["fraction_of_ceiling"])
            print(
                f"  the model that knows nothing about the line ({reference}): "
                f"{float(row['mean_r']):.4f}, {fraction:.1%} of the ceiling"
            )
    inconclusive = comparisons.loc[
        (comparisons["gene_set"] == "responding")
        & (comparisons["ci_lo"] <= 0)
        & (comparisons["ci_hi"] >= 0)
    ]
    print(
        f"\ncomparisons whose interval includes zero: {len(inconclusive)} of "
        f"{len(comparisons.loc[comparisons['gene_set'] == 'responding'])} on responding genes"
    )

## What this rung does not establish, and what to read with care

**The ceiling's known limits** (design section 6). Four, stated before the run:

1. **Averaging order.** The ceiling averages correlations over pairs and then takes the square
   root, which makes it slightly high — so every fraction of it reads slightly low.
2. **Different responding genes.** Rung 0 chose responding genes from half of a pair's plates and
   rung 1 chooses them from all of them, so the two gene sets are not identical. Rung 0's number
   is the closest ceiling that exists.
3. **Three-plate pairs.** Fifty pairs were measured on three plates; their answers are more
   reliable than the ceiling assumes, so their fraction reads slightly high.
4. **Shared wells.** All fifty lines share wells, so a well's laboratory error reaches the hidden
   line and the training lines alike. It cancels in a comparison between two models, but can
   inflate a fraction of the ceiling by an amount this task does not measure.

**The pairs a model had already seen.** The sci-Plex fine-tune was trained on A549 (`ACH-000681`)
with five compounds, two of which are in this grid. Both pairs are removed for **every** model,
not only for the fine-tune, so all models are still scored on the same pairs; the count is
printed below.

**The leave-one-drug-out stand-in rows are not a clean zero baseline.** With a drug hidden, every
line is in training, so a random stand-in of the same width already spans part of the space the
lines live in — task 9 measured it recovering about 20/49 of a planted line effect (`decisions.md`,
2026-09-11). Under that scheme the question is whether a description beats its stand-in, not
whether the stand-in sits at zero; the leave-one-line-out rows are the clean version of that test.

**Not in this task at all** (design section 12): Stack generating responses directly, the 0.05
and 0.5 micromolar doses, re-running rung 0, and hiding a line and a drug at the same time.


In [ ]:
leakage = record("rung1_leakage_profiles.json")
if leakage:
    rows = [
        {
            "model version": p["model_version"],
            "checkpoint": p["checkpoint"],
            "fine-tuned on sci-Plex": p["sciplex_fine_tuned"],
            "saw Tahoe's treated cells": p["saw_tahoe_treated_cells"],
            "pairs it had seen": p["n_exposed_pairs"],
            "the line is in this grid": p["sciplex_line_in_grid"],
        }
        for p in leakage
    ]
    display(pd.DataFrame(rows).set_index("model version"))
    print(leakage[0]["pretraining_note"])
    print(
        "\nand those pairs were removed for every model, not only for the one that saw them: "
        f"{leakage[0]['excluded_pairs_removed_for_every_model']}"
    )

## The scripts this task touched, in run order

**Data** — `scripts/heldout_grid.py` (the grid, the ceiling and the restriction record) ·
`scripts/heldout_fetch_metadata.py` · `scripts/heldout_answers.py` (the answer scan and its
combine) · `scripts/heldout_dmso_cells.py` (untreated cells, and their combine) ·
`scripts/heldout_embed.py` with `scripts/strip_ckpt_head.py` (the three Stack checkpoints) ·
`scripts/heldout_descriptions.py` (expression, its compressions, the stand-ins, drug chemistry).

**Measurement** — `scripts/heldout_fit.py` (one round: every model fitted and scored) ·
`scripts/heldout_redraws.py` (one block of redraws) · `scripts/heldout_combine.py` (the result
tables, the five control tables, the figures, the leakage records and the parameter sidecar).

**The library behind them** — `src/fmharness/heldout/`: `grid.py`, `answers.py`, `cells.py`,
`descriptions.py`, `chemistry.py`, `models.py`, `scoring.py`, `comparisons.py`, `controls.py`,
`figures.py`, `leakage.py`, `records.py`, with `src/fmharness/statistics.py` and
`src/fmharness/tahoe.py`.

**On the cluster** — `scripts/alpine/rung1_env.sh` and the job chain it is sourced by:
`rung1_grid.sbatch`, `rung1_answers.sbatch`, `rung1_answers_combine.sbatch`,
`rung1_dmso_cells.sbatch`, `rung1_dmso_combine.sbatch`, `rung1_embed.sbatch`,
`rung1_descriptions.sbatch`, `rung1_fit.sbatch`, `rung1_redraws.sbatch`, `rung1_combine.sbatch`,
submitted by `scripts/alpine/submit_rung1_chain.sh`.

**Verification** — `scripts/verify_rung1.py` and [verify.ipynb](verify.ipynb), which recompute
every claim above from the artifacts alone.

**Controls and known-answer tests** — `tests/test_rung1_controls.py`,
`tests/test_heldout_pipeline.py`, `tests/test_heldout_models.py`, `tests/test_heldout_scoring.py`,
`tests/test_heldout_descriptions.py`, `tests/test_heldout_cells.py`,
`tests/test_heldout_answers.py`, `tests/test_heldout_grid.py`, `tests/test_heldout_embed.py`,
`tests/test_rung1_jobs.py`, `tests/test_verify_rung1.py`.
